# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedsufian10/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- **Unit of analysis:** One row = one content item (webpage) aggregated over a specific month.
- **Time window:** We are using a mid-panel month (`month=2026-03`) to build our features, which leaves the final month (June 2026) safely hidden as our future test window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Features:** Past impressions, clicks, position, sessions, and age.
- **Label / proxy:** Future traffic decline (measured in the months *after* March).
- **Context:** `client_hash_id` and `content_hash_id` (used for grouping, never for the model to learn).
- **Excluded:** Any product-decision flags like `action_type` or `priority_score`. **Why?** Because these are generated by existing rules; if we use them as features, our model will just blindly memorize the old rules instead of discovering real patterns (a circular result).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

MARCH_FACTS = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("--- 1. GRAIN CHECK (should return 0 rows if grain holds) ---")
print(con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as c
    FROM {MARCH_FACTS}
    GROUP BY 1, 2, 3
    HAVING c > 1
""").df())

print("\n--- 2. ROW COUNT & DATE SPAN ---")
print(con.sql(f"""
    SELECT MIN(report_date) as start, MAX(report_date) as end, COUNT(*) as total_rows
    FROM {MARCH_FACTS}
""").df())

print("\n--- 3. AVAILABILITY CHECK ---")
print(con.sql(f"""
    SELECT COUNT(*) as total,
           COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as gsc_rows
    FROM {MARCH_FACTS}
""").df())

print("\n--- 4. FIVE FEATURES (safe) + THE TRAP (leaked column) ---")
features_df = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS feature_1_impressions,
        SUM(gsc_clicks) AS feature_2_clicks,
        AVG(gsc_avg_position) AS feature_3_position,
        SUM(ga4_sessions) AS feature_4_sessions,
        COUNT(report_date) AS feature_5_days_visible,
        SUM(gsc_impressions) * 0.8 AS trap_future_traffic_estimate
    FROM {MARCH_FACTS}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1
    LIMIT 5
""").df()

print("WITH trap column (notice how it closely mirrors impressions):")
print(features_df)

features_df = features_df.drop(columns=['trap_future_traffic_estimate'])
print("\nTrap removed! Honest features remaining:", list(features_df.columns))

--- 1. GRAIN CHECK (should return 0 rows if grain holds) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []

--- 2. ROW COUNT & DATE SPAN ---
       start        end  total_rows
0 2026-03-01 2026-03-31     9841378

--- 3. AVAILABILITY CHECK ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     total  gsc_rows
0  9841378   3611061

--- 4. FIVE FEATURES (safe) + THE TRAP (leaked column) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WITH trap column (notice how it closely mirrors impressions):
            content_hash_id  feature_1_impressions  feature_2_clicks  \
0  content_b7e512995f79d5a6                 1140.0               2.0   
1  content_05597932fe4da067                   57.0               0.0   
2  content_905aa32a0230694e                  149.0               0.0   
3  content_05434271b257bb68                 1421.0               6.0   
4  content_d056587ff7faca0c                 2770.0              16.0   

   feature_3_position  feature_4_sessions  feature_5_days_visible  \
0            4.394234                 0.0                      31   
1            2.714744                 0.0                      26   
2            6.481453                 4.0                      30   
3            6.320337                 9.0                      31   
4            4.459107                 3.0                      31   

   trap_future_traffic_estimate  
0                         912.0  
1                     

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation 1 — Unbalanced history:** Different clients started tracking data on different dates. A zero impression row early in the history might mean "no tracking yet", not "no traffic". We always filter by `gsc_data_available IS TRUE` before drawing any conclusions.

**Limitation 2 — GSC-only early rows:** Early rows in the dataset contain only Search Console data (impressions, clicks, position) with no GA4 session or engagement data. Any feature built from `ga4_sessions` will be NULL for those rows, which can silently bias the model toward clients with longer GA4 history.

**Limitation 3 — Window overlap risk:** If we define features and labels from the same time window (e.g., March features AND March decline label), the model sees the answer in the input. We deliberately use March as the feature window and later months as the target window to keep them fully separated.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.